<a href="https://colab.research.google.com/github/Adyan213/Hands-On-ML/blob/main/Chapter_12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [102]:
import tensorflow as tf

In [103]:
t=tf.constant([[1., 2., 3.], [4., 5., 6.]])
t

<tf.Tensor: shape=(2, 3), dtype=float32, numpy=
array([[1., 2., 3.],
       [4., 5., 6.]], dtype=float32)>

In [104]:
t.shape

TensorShape([2, 3])

In [105]:
t.dtype

tf.float32

In [106]:
t[:, 1:]

<tf.Tensor: shape=(2, 2), dtype=float32, numpy=
array([[2., 3.],
       [5., 6.]], dtype=float32)>

In [107]:
t[..., 1, tf.newaxis]

<tf.Tensor: shape=(2, 1), dtype=float32, numpy=
array([[2.],
       [5.]], dtype=float32)>

In [108]:
t+10

<tf.Tensor: shape=(2, 3), dtype=float32, numpy=
array([[11., 12., 13.],
       [14., 15., 16.]], dtype=float32)>

In [109]:
tf.square(t)

<tf.Tensor: shape=(2, 3), dtype=float32, numpy=
array([[ 1.,  4.,  9.],
       [16., 25., 36.]], dtype=float32)>

In [110]:
t @ tf.transpose(t)

<tf.Tensor: shape=(2, 2), dtype=float32, numpy=
array([[14., 32.],
       [32., 77.]], dtype=float32)>

In [111]:
tf.constant(42)

<tf.Tensor: shape=(), dtype=int32, numpy=42>

In [112]:
tf.square(tf.transpose(t))+10

<tf.Tensor: shape=(3, 2), dtype=float32, numpy=
array([[11., 26.],
       [14., 35.],
       [19., 46.]], dtype=float32)>

In [113]:
import numpy as np
a=np.array([2., 4., 5.])
tf.constant(a)

<tf.Tensor: shape=(3,), dtype=float64, numpy=array([2., 4., 5.])>

In [114]:
t.numpy()

array([[1., 2., 3.],
       [4., 5., 6.]], dtype=float32)

In [115]:
np.array(t)

array([[1., 2., 3.],
       [4., 5., 6.]], dtype=float32)

In [116]:
tf.square(a)

<tf.Tensor: shape=(3,), dtype=float64, numpy=array([ 4., 16., 25.])>

In [117]:
np.square(t)

array([[ 1.,  4.,  9.],
       [16., 25., 36.]], dtype=float32)

In [118]:
t2=tf.constant(40., dtype=tf.float64)
tf.constant(2.)+tf.cast(t2, tf.float32)

<tf.Tensor: shape=(), dtype=float32, numpy=42.0>

In [119]:
v=tf.Variable([[1., 2., 3.], [4., 5., 6.]])
v

<tf.Variable 'Variable:0' shape=(2, 3) dtype=float32, numpy=
array([[1., 2., 3.],
       [4., 5., 6.]], dtype=float32)>

In [120]:
v.assign(2*v)

<tf.Variable 'UnreadVariable' shape=(2, 3) dtype=float32, numpy=
array([[ 2.,  4.,  6.],
       [ 8., 10., 12.]], dtype=float32)>

In [121]:
v[0, 1].assign(42)

<tf.Variable 'UnreadVariable' shape=(2, 3) dtype=float32, numpy=
array([[ 2., 42.,  6.],
       [ 8., 10., 12.]], dtype=float32)>

In [122]:
v[:, 2].assign([0., 1.])

<tf.Variable 'UnreadVariable' shape=(2, 3) dtype=float32, numpy=
array([[ 2., 42.,  0.],
       [ 8., 10.,  1.]], dtype=float32)>

In [123]:
v.scatter_nd_update(
    indices=[[0, 0], [1, 2]],
    updates=[100., 200.]
)

<tf.Variable 'UnreadVariable' shape=(2, 3) dtype=float32, numpy=
array([[100.,  42.,   0.],
       [  8.,  10., 200.]], dtype=float32)>

In [124]:
def huber_fn(y_true, y_pred):
  error=y_true-y_pred
  is_small_error=tf.abs(error)<1
  squared_loss=tf.square(error)/2
  linear_loss=tf.abs(error)-0.5
  return tf.where(is_small_error, squared_loss, linear_loss)

In [125]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

housing = fetch_california_housing()
X_train_full, X_test, y_train_full, y_test = train_test_split(
    housing.data, housing.target.reshape(-1, 1), random_state=42)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full, y_train_full, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)
X_test_scaled = scaler.transform(X_test)

input_shape = X_train.shape[1:]

tf.keras.utils.set_random_seed(42)
model = tf.keras.Sequential([
    tf.keras.layers.Dense(30, activation="relu", kernel_initializer="he_normal",
                          input_shape=input_shape),
    tf.keras.layers.Dense(1),
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [126]:
model.compile(loss=huber_fn, optimizer='nadam', metrics=['mae'])

In [127]:
model.fit(X_train_scaled, y_train, epochs=2,
        validation_data=(X_valid_scaled, y_valid))

Epoch 1/2
363/363 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.4838 - mae: 0.8334 - val_loss: 0.3474 - val_mae: 0.6522
Epoch 2/2
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.2404 - mae: 0.5406 - val_loss: 0.2553 - val_mae: 0.5383


In [128]:
model.save("my_model_with_a_custom_loss.keras")

In [129]:
model=tf.keras.models.load_model('my_model_with_a_custom_loss.keras',
                                 custom_objects={'huber_fn':huber_fn})

In [130]:
def create_huber(threshold=1.0):
    def huber_fn(y_true, y_pred):
        error = y_true - y_pred
        is_small_error = tf.abs(error) < threshold
        squared_loss = tf.square(error) / 2
        linear_loss  = threshold * tf.abs(error) - threshold ** 2 / 2
        return tf.where(is_small_error, squared_loss, linear_loss)
    return huber_fn

In [131]:
model.fit(X_train_scaled, y_train, epochs=2,
        validation_data=(X_valid_scaled, y_valid))

Epoch 1/2
363/363 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.2065 - mae: 0.4930 - val_loss: 0.2129 - val_mae: 0.4876
Epoch 2/2
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.1904 - mae: 0.4700 - val_loss: 0.1891 - val_mae: 0.4610


In [132]:
class HuberLoss(tf.keras.losses.Loss):
  def __init__(self, threshold=1.0, **kwargs):
    self.threshold=threshold
    super().__init__(**kwargs)

  def call(self, y_true, y_pred):
    error=y_true-y_pred
    is_samll_error=tf.abs(error)<self.threshold
    squared_loss=tf.square(error)/2
    linear_loss=self.threshold*tf.abs(error)-self.threshold**2/2
    return tf.where(is_samll_error, squared_loss, linear_loss)

  def get_config(self):
    base_config=super().get_config()
    return {**base_config, 'threshold':self.threshold}

In [133]:
tf.keras.utils.set_random_seed(42)
model = tf.keras.Sequential([
    tf.keras.layers.Dense(30, activation="relu", kernel_initializer="he_normal",
                          input_shape=input_shape),
    tf.keras.layers.Dense(1),
])

In [134]:
model.compile(loss=HuberLoss(2.), optimizer="nadam", metrics=["mae"])

In [135]:
model.fit(X_train_scaled, y_train, epochs=2,
          validation_data=(X_valid_scaled, y_valid))

Epoch 1/2
363/363 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.6461 - mae: 0.8446 - val_loss: 0.5086 - val_mae: 0.6718
Epoch 2/2
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.2899 - mae: 0.5544 - val_loss: 0.3527 - val_mae: 0.5571


In [136]:
model.save("my_model_with_a_custom_loss_class.keras")

In [137]:
model=tf.keras.models.load_model('my_model_with_a_custom_loss_class.keras',
                                 custom_objects={'HuberLoss':HuberLoss})

In [138]:
model.fit(X_train_scaled, y_train, epochs=2,
          validation_data=(X_valid_scaled, y_valid))

Epoch 1/2
363/363 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 0.2444 - mae: 0.5073 - val_loss: 0.2689 - val_mae: 0.4982
Epoch 2/2
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.2213 - mae: 0.4818 - val_loss: 0.2207 - val_mae: 0.4657


In [139]:
model.loss.threshold

2.0

In [140]:
def my_softplus(z):
  return tf.math.log(1.0+tf.exp(z))

def my_glorot_initializer(shape, dtype=tf.float32):
  stddev=tf.sqrt(2.0/(shape[0]+shape[1]))
  return tf.random.normal(shape, stddev=stddev, dtype=dtype)

def my_l1_regularizer(weights):
  return tf.reduce_sum(tf.abs(0.01*weights))

def my_positive_weights(weights):
  return tf.where(weights<0., tf.zeros_like(weights), weights)

In [141]:
layer=tf.keras.layers.Dense(1, activation=my_softplus,
                            kernel_initializer=my_glorot_initializer,
                            kernel_regularizer=my_l1_regularizer,
                            kernel_constraint=my_positive_weights)

In [142]:
class MyL1Regularizer(tf.keras.regularizers.Regularizer):
  def __init__(self, factor):
    self.factor=factor
  def __call__(self, weights):
    return tf.reduce_sum(tf.abs(self.factor*weights))

  def get_config(self):
    return {'factor':self.factor}

In [143]:
tf.keras.utils.set_random_seed(42)
model = tf.keras.Sequential([
    tf.keras.layers.Dense(30, activation="relu", kernel_initializer="he_normal",
                          input_shape=input_shape),
    tf.keras.layers.Dense(1),
])

In [144]:
model.compile(loss="mse", optimizer="nadam", metrics=[HuberLoss(2.0)])

In [145]:
model.fit(X_train_scaled, y_train, epochs=2)

Epoch 1/2
363/363 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - huber_loss_3: 0.6813 - loss: 1.7395
Epoch 2/2
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - huber_loss_3: 0.3132 - loss: 0.7699


In [146]:
precision=tf.keras.metrics.Precision()
precision([0, 1, 1, 1, 0, 1, 0, 1], [1, 1, 0, 1, 0, 1, 0, 1])

<tf.Tensor: shape=(), dtype=float32, numpy=0.800000011920929>

In [147]:
precision([0, 1, 0, 0, 1, 0, 1, 1], [1, 0, 1, 1, 0, 0, 0, 0])

<tf.Tensor: shape=(), dtype=float32, numpy=0.5>

In [148]:
precision.result()

<tf.Tensor: shape=(), dtype=float32, numpy=0.5>

In [149]:
precision.variables

[<Variable path=precision_1/true_positives, shape=(1,), dtype=float32, value=[4.]>,
 <Variable path=precision_1/false_positives, shape=(1,), dtype=float32, value=[4.]>]

In [150]:
precision.reset_state()

In [151]:
class HuberMetric(tf.keras.metrics.Metric):
  def __init__(self, threshold=1.0, **kwargs):
    super().__init__(**kwargs)
    self.threshold=threshold
    self.huber_fn=create_huber(threshold)
    self.total=self.add_weight(name='total', initializer='zeros')
    self.count=self.add_weight(name='count', initializer='zeros')

  def update_state(self, y_true, y_pred, sample_weight=None):
    sample_metrics=self.huber_fn(y_true, y_pred)
    self.total.assign_add(tf.reduce_sum(sample_metrics))
    self.count.assign_add(tf.cast(tf.size(y_true), tf.float32))

  def result(self):
    return self.total/self.count

  def get_config(self):
    base_config = super().get_config()
    return {**base_config, "threshold": self.threshold}

In [152]:
tf.keras.utils.set_random_seed(42)
model = tf.keras.Sequential([
    tf.keras.layers.Dense(30, activation="relu", kernel_initializer="he_normal",
                          input_shape=input_shape),
    tf.keras.layers.Dense(1),
])
model.compile(loss=create_huber(2.0), optimizer="nadam",
              metrics=[HuberMetric(2.0)])
model.fit(X_train_scaled, y_train, epochs=2)


Epoch 1/2
363/363 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - huber_metric_1: 0.6461 - loss: 0.6461
Epoch 2/2
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - huber_metric_1: 0.2899 - loss: 0.2899


In [153]:
model.save("my_model_with_a_custom_metric.keras")

In [154]:
model = tf.keras.models.load_model(
    "my_model_with_a_custom_metric.keras",
    custom_objects={
        "huber_fn": create_huber(2.0),
        "HuberMetric": HuberMetric
    }
)

In [155]:
model.fit(X_train_scaled, y_train, epochs=2)

Epoch 1/2
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - huber_metric_1: 0.2444 - loss: 0.2444
Epoch 2/2
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - huber_metric_1: 0.2213 - loss: 0.2213


In [156]:
exponential_layer=tf.keras.layers.Lambda(lambda x: tf.exp(x))

In [157]:
class MyDense(tf.keras.layers.Layer):
  def __init__(self, units, activation=None, **kwargs):
    super().__init__(**kwargs)
    self.units=units
    self.activation=tf.keras.activations.get(activation)

  def build(self, batch_input_shape):
    self.kernel=self.add_weight(
        name='kernel', shape=[batch_input_shape[-1], self.units],
        initializer='glorot_normal'
    )
    self.bias=self.add_weight(
        name='bias', shape=[self.units], initializer='zeros'
    )

  def call(self, X):
    return self.activation(X@self.kernel+self.bias)

  def get_config(self):
    base_config=super().get_config()
    return {**base_config, 'units':self.units,
            'activation':tf.keras.activations.serialize(self.activation)}

In [158]:
tf.keras.utils.set_random_seed(42)
model = tf.keras.Sequential([
    MyDense(30, activation="relu", input_shape=input_shape),
    MyDense(1)
])
model.compile(loss="mse", optimizer="nadam")
model.fit(X_train_scaled, y_train, epochs=2,
          validation_data=(X_valid_scaled, y_valid))
model.evaluate(X_test_scaled, y_test)
model.save("my_model_with_a_custom_layer.keras")

Epoch 1/2


/tmp/ipykernel_2626/458593097.py:3: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


363/363 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 2.3247 - val_loss: 1.7553
Epoch 2/2
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.6376 - val_loss: 0.6906
162/162 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.5459


In [159]:
model = tf.keras.models.load_model("my_model_with_a_custom_layer.keras",
                                   custom_objects={"MyDense": MyDense})
model.fit(X_train_scaled, y_train, epochs=2,
          validation_data=(X_valid_scaled, y_valid))

Epoch 1/2
363/363 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.5072 - val_loss: 0.4317
Epoch 2/2
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.4474 - val_loss: 0.4608


In [160]:
class ResidualBlock(tf.keras.layers.Layer):
  def __init__(self, n_layers, n_neurons, **kwargs):
    super().__init__(**kwargs)
    self.hidden=[tf.keras.layer.Dense(n_neurons, activation='relu',
                                      kernel_initializer='he_normal')
                for _ in range(n_layers)]
  def call(self, inputs):
    z=inputs
    for layer in self.hidden:
      z=layer(z)
    return inputs+z

In [161]:
class ResidualRegressor(tf.keras.Model):
  def __init__(self, output_dim, **kwargs):
    super().__init__(**kwargs)
    self.hidden1=tf.keras.layers.Dense(30, activation='relu',
                                      kernel_initializer='he_normal')
    self.block1=ResidualBlock(2, 30)
    self.block2=ResidualBlock(2, 30)
    self.out=tf.keras.layers.Dense(output_dim)

  def call(self, inputs):
    z=self.hidden1(inputs)
    for _ in range(1+3):
      z=self.block1(z)
    z=self.block2(z)
    return self.out

  def get_config(self):
        base_config = super().get_config()
        return {**base_config, "output_dim": self.output_dim}

In [162]:
class ReconstructingRegressor(tf.keras.Model):
  def __init__(self, output_dim, **kwargs):
    super().__init__(**kwargs)
    self.hidden=[tf.keras.layers.Dense(30, activation='relu',
                                       kernel_initializer='he_normal')
                for _ in range(5)]
    self.out=tf.keras.layers.Dense(output_dim)
    self.reconstruction_mean=tf.keras.metrics.Mean(name='reconstruction_error')

  def build(self, batch_input_shape):
    n_inputs=batch_input_shape[-1]
    self.reconstruct=tf.keras.layers.Dense(n_inputs)

  def call(self, inputs, training=None):
    z=inputs
    for layer in self.hidden:
      z=layer(z)
    reconstruction=self.reconstruct(z)
    recon_loss=tf.reduce_mean(tf.square(reconstruction-inputs))
    self.add_loss(0.05*recon_loss)
    return self.out(z)

In [163]:
tf.keras.utils.set_random_seed(42)
model = ReconstructingRegressor(1)
model.compile(loss="mse", optimizer="nadam")
history = model.fit(X_train_scaled, y_train, epochs=5)
y_pred = model.predict(X_test_scaled)

Epoch 1/5
363/363 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 0.7571 - reconstruction_error: 0.0000e+00
Epoch 2/5
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.5037 - reconstruction_error: 0.0000e+00
Epoch 3/5
363/363 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.4193 - reconstruction_error: 0.0000e+00
Epoch 4/5
363/363 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.4187 - reconstruction_error: 0.0000e+00
Epoch 5/5
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.3705 - reconstruction_error: 0.0000e+00
162/162 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


In [164]:
def f(w1, w2):
  return 3*w1**2+2*w1*w2

In [165]:
w1, w2=5, 3
eps=1e-6
(f(w1+eps, w2)-f(w1, w2))/eps

36.000003007075065

In [166]:
(f(w1, w2+eps)-f(w1, w2))/eps

10.000000003174137

In [167]:
w1, w2 = tf.Variable(5.), tf.Variable(3.)
with tf.GradientTape() as tape:
  z=f(w1, w2)

gradients=tape.gradient(z, [w1, w2])

In [168]:
gradients

[<tf.Tensor: shape=(), dtype=float32, numpy=36.0>,
 <tf.Tensor: shape=(), dtype=float32, numpy=10.0>]

In [169]:
with tf.GradientTape() as tape:
    z = f(w1, w2)

dz_dw1 = tape.gradient(z, w1)
try:
    dz_dw2 = tape.gradient(z, w2)
except RuntimeError as ex:
    print(ex)

A non-persistent GradientTape can only be used to compute one set of gradients (or jacobians)


In [170]:
with tf.GradientTape(persistent=True) as tape:
    z = f(w1, w2)

dz_dw1 = tape.gradient(z, w1)
dz_dw2 = tape.gradient(z, w2)
del tape

In [171]:
c1, c2 = tf.constant(5.), tf.constant(3.)
with tf.GradientTape() as tape:
    z = f(c1, c2)

gradients = tape.gradient(z, [c1, c2])

In [172]:
gradients

[None, None]

In [173]:
with tf.GradientTape() as tape:
    tape.watch(c1)
    tape.watch(c2)
    z = f(c1, c2)

gradients = tape.gradient(z, [c1, c2])
gradients

[<tf.Tensor: shape=(), dtype=float32, numpy=36.0>,
 <tf.Tensor: shape=(), dtype=float32, numpy=10.0>]

In [174]:
def f(w1, w2):
    return 3 * w1 ** 2 + tf.stop_gradient(2 * w1 * w2)

with tf.GradientTape() as tape:
    z = f(w1, w2)  # same result as without stop_gradient()

gradients = tape.gradient(z, [w1, w2])
gradients

[<tf.Tensor: shape=(), dtype=float32, numpy=30.0>, None]

In [175]:
x=tf.Variable(1e-50)
with tf.GradientTape() as tape:
  z=tf.sqrt(x)
tape.gradient(z, [x])

[<tf.Tensor: shape=(), dtype=float32, numpy=inf>]

In [176]:
def my_softplus(z):
    return tf.math.log(1 + tf.exp(-tf.abs(z))) + tf.maximum(0., z)

In [177]:
@tf.custom_gradient
def my_softplus(z):
    def my_softplus_gradients(grads):  # grads = backprop'ed from upper layers
        return grads * (1 - 1 / (1 + tf.exp(z)))  # stable grads of softplus

    result = tf.math.log(1 + tf.exp(-tf.abs(z))) + tf.maximum(0., z)
    return result, my_softplus_gradients

In [178]:
x = tf.Variable([1000.])
with tf.GradientTape() as tape:
    z = my_softplus(x)

z, tape.gradient(z, [x])

(<tf.Tensor: shape=(1,), dtype=float32, numpy=array([1000.], dtype=float32)>,
 [<tf.Tensor: shape=(1,), dtype=float32, numpy=array([1.], dtype=float32)>])

In [179]:
def cube(x):
  return x**3

In [180]:
cube(2)

8

In [181]:
cube(tf.constant(2.0))

<tf.Tensor: shape=(), dtype=float32, numpy=8.0>

In [182]:
tf_cube=tf.function(cube)
tf_cube

In [183]:
tf_cube(2)

<tf.Tensor: shape=(), dtype=int32, numpy=8>

In [184]:
tf_cube(tf.constant(2.0))

<tf.Tensor: shape=(), dtype=float32, numpy=8.0>

In [185]:
@tf.function
def tf_cube(x):
  return x**3

In [186]:
class LayerNormalization(tf.keras.layers.Layer):
  def __init__(self, eps=0.001, **kwargs):
    super().__init__(**kwargs)
    self.eps=eps

  def build(self, batch_input_shape):
    self.alpha=self.add_weight(
        name='alpha', shape=batch_input_shape[-1:],
        initializer='ones'
    )
    self.beta=self.add_weight(
        name='beta', shape=batch_input_shape[-1:],
        initializer='zeros'
    )

  def call(self, X):
    mean, variance = tf.nn.moments(X, axes=-1, keepdims=True)
    return self.alpha*(X-mean)/(tf.sqrt(variance+eps))+self.beta

  def get_config(self):
    base_config=super().get_config()
    return {**base_config, 'eps':self.eps}

In [187]:
X=X_train.astype(np.float32)

custom_layer=LayerNormalization()
keras_layer=tf.keras.layers.LayerNormalization()
keras_layer(X)


<tf.Tensor: shape=(11610, 8), dtype=float32, numpy=
array([[-0.35116717, -0.3274265 , -0.35214227, ..., -0.3551287 ,
        -0.2806219 , -0.6116668 ],
       [-0.36530852, -0.36559346, -0.36429703, ..., -0.36694786,
        -0.34063148, -0.47207996],
       [-0.35567376, -0.2972846 , -0.34565884, ..., -0.3575881 ,
        -0.27600297, -0.6399096 ],
       ...,
       [-0.3660983 , -0.3137064 , -0.3620506 , ..., -0.3634033 ,
        -0.31510064, -0.55227804],
       [-0.36498532, -0.25498438, -0.35821036, ..., -0.36317444,
        -0.28858358, -0.6337247 ],
       [-0.34895486, -0.28748363, -0.34746027, ..., -0.35637128,
        -0.27044445, -0.6582793 ]], dtype=float32)>

In [188]:
custom_layer(X)

<tf.Tensor: shape=(11610, 8), dtype=float32, numpy=
array([[-0.35116723, -0.32742652, -0.3521423 , ..., -0.35512877,
        -0.2806219 , -0.61166686],
       [-0.36530855, -0.36559352, -0.3642971 , ..., -0.36694792,
        -0.3406315 , -0.47208   ],
       [-0.3556738 , -0.2972846 , -0.34565887, ..., -0.3575881 ,
        -0.27600297, -0.6399097 ],
       ...,
       [-0.36609834, -0.31370643, -0.3620506 , ..., -0.36340332,
        -0.31510064, -0.55227804],
       [-0.3649853 , -0.25498438, -0.35821036, ..., -0.3631744 ,
        -0.28858355, -0.63372463],
       [-0.34895486, -0.2874836 , -0.34746027, ..., -0.35637128,
        -0.27044445, -0.65827936]], dtype=float32)>

In [189]:
tf.reduce_mean(tf.keras.losses.MeanAbsoluteError()(keras_layer(X), custom_layer(X)))

<tf.Tensor: shape=(), dtype=float32, numpy=3.579861740377055e-08>